In [1]:
import pandas as pd

games = pd.read_csv("../data/games_raw.csv", parse_dates=["GAME_DATE"])
print(games.shape)
games.columns.tolist()

(26418, 29)


['SEASON_ID',
 'TEAM_ID',
 'TEAM_ABBREVIATION',
 'TEAM_NAME',
 'GAME_ID',
 'GAME_DATE',
 'MATCHUP',
 'WL',
 'MIN',
 'PTS',
 'FGM',
 'FGA',
 'FG_PCT',
 'FG3M',
 'FG3A',
 'FG3_PCT',
 'FTM',
 'FTA',
 'FT_PCT',
 'OREB',
 'DREB',
 'REB',
 'AST',
 'STL',
 'BLK',
 'TOV',
 'PF',
 'PLUS_MINUS',
 'SEASON']

In [2]:
games["HOME"] = games["MATCHUP"].str.contains("vs.").astype(int)  # 1 = à domicile
games["WIN"] = (games["WL"] == "W").astype(int)                    # 1 = victoire
games[["GAME_DATE", "TEAM_ABBREVIATION", "MATCHUP", "HOME", "WL", "WIN"]].head()

,GAME_DATE,TEAM_ABBREVIATION,MATCHUP,HOME,WL,WIN
0,2016-04-13,LAL,LAL vs. UTA,1,W,1
1,2016-04-13,DAL,DAL vs. SAS,1,L,0
2,2016-04-13,WAS,WAS vs. ATL,1,W,1
3,2016-04-13,BOS,BOS vs. MIA,1,W,1
4,2016-04-13,CLE,CLE vs. DET,1,L,0


In [3]:
home_win_rate = games[games["HOME"] == 1].groupby("SEASON")["WIN"].mean().round(3)
home_win_rate

SEASON
2015-16    0.589
2016-17    0.584
2017-18    0.579
2018-19    0.593
2019-20    0.551
2020-21    0.544
2021-22    0.544
2022-23    0.580
2023-24    0.543
2024-25    0.544
2025-26    0.554
Name: WIN, dtype: float64

In [4]:
# Trier chaque équipe dans l'ordre chronologique (indispensable pour shift et rolling)
games = games.sort_values(["TEAM_ID", "GAME_DATE"]).reset_index(drop=True)
grp = games.groupby(["TEAM_ID", "SEASON"])

# 1. "Classement" : % de victoires dans la saison AVANT ce match
games["WIN_PCT_SEASON"] = grp["WIN"].transform(lambda s: s.shift(1).expanding().mean())

# 2. Dynamique : % de victoires sur les 10 derniers matchs
games["WIN_PCT_LAST10"] = grp["WIN"].transform(lambda s: s.shift(1).rolling(10, min_periods=1).mean())

# 3. Domination : écart de points moyen sur les 10 derniers matchs
games["PM_LAST10"] = grp["PLUS_MINUS"].transform(lambda s: s.shift(1).rolling(10, min_periods=1).mean())

In [5]:
cols = ["GAME_DATE", "MATCHUP", "WIN", "PLUS_MINUS", "WIN_PCT_SEASON", "WIN_PCT_LAST10", "PM_LAST10"]
games[(games["TEAM_ABBREVIATION"] == "LAL") & (games["SEASON"] == "2024-25")][cols].head(6)

,GAME_DATE,MATCHUP,WIN,PLUS_MINUS,WIN_PCT_SEASON,WIN_PCT_LAST10,PM_LAST10
9515,2024-10-22,LAL vs. MIN,1,7.0,NaN,NaN,NaN
9516,2024-10-25,LAL vs. PHX,1,7.0,1.00,1.00,7.0
9517,2024-10-26,LAL vs. SAC,1,4.0,1.00,1.00,7.0
9518,2024-10-28,LAL @ PHX,0,-4.0,1.00,1.00,6.0
9519,2024-10-30,LAL @ CLE,0,-24.0,0.75,0.75,3.5
9520,2024-11-01,LAL @ TOR,1,6.0,0.60,0.60,-2.0


In [6]:
# Jours de repos depuis le dernier match de l'équipe
games["REST_DAYS"] = grp["GAME_DATE"].diff().dt.days
games["REST_DAYS"] = games["REST_DAYS"].clip(upper=4)   # au-delà de 4 jours, l'effet ne change plus
games["B2B"] = (games["REST_DAYS"] == 1).astype(int)     # 1 = joue deux soirs de suite

In [7]:
features = ["WIN_PCT_SEASON", "WIN_PCT_LAST10", "PM_LAST10", "REST_DAYS", "B2B"]

home = games[games["HOME"] == 1][["GAME_ID", "GAME_DATE", "SEASON", "TEAM_ABBREVIATION", "WIN"] + features]
away = games[games["HOME"] == 0][["GAME_ID", "TEAM_ABBREVIATION"] + features]

matches = home.merge(away, on="GAME_ID", suffixes=("_HOME", "_AWAY"))
matches = matches.rename(columns={"WIN": "HOME_WIN"})   # notre cible : 1 si l'équipe à domicile gagne
matches = matches.dropna().sort_values("GAME_DATE").reset_index(drop=True)

print(matches.shape)
matches.head()

(13023, 16)


,GAME_ID,GAME_DATE,SEASON,TEAM_ABBREVIATION_HOME,HOME_WIN,WIN_PCT_SEASON_HOME,WIN_PCT_LAST10_HOME,PM_LAST10_HOME,REST_DAYS_HOME,B2B_HOME,TEAM_ABBREVIATION_AWAY,WIN_PCT_SEASON_AWAY,WIN_PCT_LAST10_AWAY,PM_LAST10_AWAY,REST_DAYS_AWAY,B2B_AWAY
0,21500018,2015-10-29,2015-16,IND,0,0.0,0.0,-7.0,1.0,1,MEM,0.0,0.0,-30.0,1.0,1
1,21500020,2015-10-29,2015-16,LAC,1,1.0,1.0,7.0,1.0,1,DAL,1.0,1.0,16.0,1.0,1
2,21500019,2015-10-29,2015-16,NYK,0,1.0,1.0,25.0,1.0,1,ATL,0.0,0.0,-12.0,2.0,0
3,21500025,2015-10-30,2015-16,DET,1,1.0,1.0,8.5,2.0,0,CHI,1.0,1.0,8.5,2.0,0
4,21500027,2015-10-30,2015-16,MIL,0,0.0,0.0,-25.0,2.0,0,WAS,1.0,1.0,1.0,2.0,0


In [8]:
matches.to_csv("../data/matches.csv", index=False)